In [ ]:
!pip install unsloth

# Import Library

In [ ]:
import unsloth

# LLM model training
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from transformers import TrainingArguments, DataCollatorForSeq2Seq, TextStreamer
from trl import SFTTrainer
from datasets import load_dataset
# Saving model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Setting up the model

In [ ]:
base_model = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
dataset_name = '/kaggle/input/datasets/cngchis/medical-qa-data'
new_model = 'Llama-3.1-8B-Instruct-Chat-Doctor'

In [ ]:
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# Prepare Data

## Load dataset

In [ ]:
dataset = load_dataset(dataset_name, split="all")

## Format Chat Template

In [ ]:
instruction = """Bạn là một bác sĩ chăm sóc khách hàng tên Chis.
    Hãy lịch sự với khách hàng và trả lời tất cả các câu hỏi của họ.
    """
def format_chat_template(row):

    row_json = [{"role": "system", "content": instruction },
               {"role": "user", "content": row["question"]},
               {"role": "assistant", "content": row["answer"]}]

    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    num_proc= 4,
)

dataset

In [ ]:
dataset['text'][3]

## Split dataset

In [ ]:
split_1 = dataset.train_test_split(test_size=0.1)
split_2 = split_1["train"].train_test_split(test_size=0.1667)

train_dataset = split_2["train"] # 75%
val_dataset   = split_2["test"] # 15%
test_dataset  = split_1["test"] # 10%

# Baseline inference

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": instruction},
    {"role": "user", "content": "Chào bác sĩ, Em vừa mới sinh bé được 3 tháng. Do em bị viêm xoang nên gần đây hay bị sổ mũi, ho và cứ đêm thức cho bé bú em lại bị như vậy (bé bú mẹ hoàn toàn). Bác sĩ cho em hỏi viêm xoang, sổ mũi sau sinh khắc phục thế nào? Có cách nào giảm triệu chứng sổ mũi, ho của em mà không cần dùng thuốc? Nếu phải dùng thuốc thì thuốc nào sẽ ít hoặc không ảnh hưởng đến bé? Bé sẽ bị ảnh hưởng như thế nào nếu em dùng thuốc tây? Em cảm ơn bác sĩ."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors='pt').to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

# Model Trainning

In [ ]:
#Hyperparamter
training_arguments = TrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs = 1,
    eval_strategy="steps",
    eval_steps=0.2,
    warmup_steps = 10,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    remove_unused_columns = False,
    average_tokens_across_devices=False
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer = tokenizer,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = training_arguments,
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
trainer_stats = trainer.train()

# Inference

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [{"role": "system", "content": instruction},
    {"role": "user", "content": "Chào bác sĩ, Em vừa mới sinh bé được 3 tháng. Do em bị viêm xoang nên gần đây hay bị sổ mũi, ho và cứ đêm thức cho bé bú em lại bị như vậy (bé bú mẹ hoàn toàn). Bác sĩ cho em hỏi viêm xoang, sổ mũi sau sinh khắc phục thế nào? Có cách nào giảm triệu chứng sổ mũi, ho của em mà không cần dùng thuốc? Nếu phải dùng thuốc thì thuốc nào sẽ ít hoặc không ảnh hưởng đến bé? Bé sẽ bị ảnh hưởng như thế nào nếu em dùng thuốc tây? Em cảm ơn bác sĩ."}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text.split("assistant")[1])

In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                   streamer = text_streamer, max_new_tokens = 512)

# Saving the tokenizer and model

In [ ]:
model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)